# Query galaxy redshifts from RA and Dec

Input galaxy-centre coordinates (ICRS, decimal degrees), search NED and SDSS, and inspect redshift candidates. No cutouts, HostPhot or SED fitting required.

- **NED:** published redshift summary plus an optional detailed measurements/reference table. A NED redshift is **not automatically spectroscopic**.
- **SDSS DR17:** spectroscopic candidates, formal redshift errors, spectral class and zWarning. Survey coverage is incomplete.
- Every candidate has an angular separation; no redshift is adopted automatically. Repeat spectra and duplicate NED/SDSS entries are retained, not counted as independent galaxies.

This first version does not query Legacy Survey or Rubin photo-z catalogs. No match means no candidate from these services within the chosen radius, not that no redshift exists anywhere.

**Validation:** code syntax checked; network queries were not executed during notebook creation. Service availability and schema changes can affect results.

## 1. Install in your notebook environment
Uncomment the next cell if needed, run it, then restart the kernel. No heavy SED packages are needed.

In [ ]:
# %pip install -U astroquery astropy pandas numpy


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from importlib.metadata import version
import json
import time
import warnings
import numpy as np
import pandas as pd
import astropy.units as u
from astropy.coordinates import SkyCoord
from astroquery.ipac.ned import Ned
from astroquery.sdss import SDSS
from IPython.display import display

Ned.TIMEOUT = 300
SDSS.TIMEOUT = 60
OUT = Path('galaxy_redshift_results')
OUT.mkdir(exist_ok=True)
print('astroquery:', version('astroquery'))


## 2. Your galaxy
Replace the example coordinates below with the galaxy centre, not the transient position. The example is approximately M87 and is only for demonstration. Start with 3 arcsec; increase cautiously for extended galaxies or uncertain centres. SDSS cone searches are limited to 3 arcmin.

In [ ]:
SN_ID = 'SN2026fgl'
RA = 149.218165    # replace with your galaxy RA, degrees
DEC = 	 3.469919    # replace with your galaxy Dec, degrees
RADIUS_ARCSEC = 3.0
SDSS_RELEASE = 17


## 3. Query functions
Raw Astropy tables are returned alongside the normalized candidates. The parser handles both decimal-degree and sexagesimal NED coordinate columns, and common old/new redshift column names. An unexpected schema raises a visible error rather than returning a misleading no-match.

Each service gets an independent status (`ok`, `no_match`, or `error`); an outage is never reported as a successful empty search. NED rows without a redshift remain visible. Negative measured redshifts are retained.

In [ ]:
COLUMNS = ['source', 'source_id', 'ra', 'dec', 'sep_arcsec', 'z', 'z_err',
           'z_kind', 'object_type', 'z_warning', 'quality_note']

def text_value(value):
    if value is None or np.ma.is_masked(value):
        return ''
    return value.decode().strip() if isinstance(value, bytes) else str(value).strip()

def number(value):
    if value is None or np.ma.is_masked(value):
        return np.nan
    try:
        return float(value)
    except (TypeError, ValueError):
        return np.nan

def find_column(table, *names, required=True):
    lookup = {str(k).lower(): k for k in table.colnames}
    for name in names:
        if name.lower() in lookup:
            return lookup[name.lower()]
    if required:
        raise KeyError(f'Expected one of {names}; received {table.colnames}')
    return None

def get_value(row, key):
    return None if key is None else row[key]

def ned_candidates(table, center):
    if table is None or len(table) == 0:
        return pd.DataFrame(columns=COLUMNS)
    ra_key = find_column(table, 'RA', 'RA(deg)', 'RA-s')
    dec_key = find_column(table, 'DEC', 'DEC(deg)', 'Dec-s')
    z_key = find_column(table, 'Redshift', 'Redshift(z)', 'z')
    name_key = find_column(table, 'Object Name', 'ObjectName')
    type_key = find_column(table, 'Type', 'Object Type', required=False)
    flag_key = find_column(table, 'Redshift Flag', 'zflag', required=False)
    rows = []
    for row in table:
        if ra_key.lower().endswith('-s'):
            pos = SkyCoord(text_value(row[ra_key]), text_value(row[dec_key]),
                           unit=(u.hourangle, u.deg), frame='icrs')
        else:
            pos = SkyCoord(number(row[ra_key])*u.deg, number(row[dec_key])*u.deg)
        rows.append(dict(source='NED', source_id=text_value(row[name_key]),
                         ra=pos.ra.deg, dec=pos.dec.deg,
                         sep_arcsec=center.separation(pos).arcsec,
                         z=number(row[z_key]), z_err=np.nan,
                         z_kind='unverified_literature',
                         object_type=text_value(get_value(row, type_key)),
                         z_warning=text_value(get_value(row, flag_key)),
                         quality_note='Inspect NED redshift measurements and reference.'))
    return pd.DataFrame(rows, columns=COLUMNS)

def sdss_candidates(table, center):
    if table is None or len(table) == 0:
        return pd.DataFrame(columns=COLUMNS)
    keys = {k: find_column(table, k) for k in
            ['ra', 'dec', 'specobjid', 'z', 'zErr', 'zWarning', 'class']}
    rows = []
    for row in table:
        pos = SkyCoord(number(row[keys['ra']])*u.deg, number(row[keys['dec']])*u.deg)
        flag = number(row[keys['zWarning']])
        rows.append(dict(source=f'SDSS_DR{SDSS_RELEASE}',
                         source_id=text_value(row[keys['specobjid']]),
                         ra=pos.ra.deg, dec=pos.dec.deg,
                         sep_arcsec=center.separation(pos).arcsec,
                         z=number(row[keys['z']]), z_err=number(row[keys['zErr']]),
                         z_kind='spectroscopic', object_type=text_value(row[keys['class']]),
                         z_warning=flag,
                         quality_note=('zWarning=0; still verify object identity.' if flag == 0
                                       else 'Flagged or missing zWarning: inspect spectrum.')))
    return pd.DataFrame(rows, columns=COLUMNS)

def query_redshifts(ra, dec, radius_arcsec=3.0):
    ra, dec, radius_arcsec = float(ra), float(dec), float(radius_arcsec)
    if not (np.isfinite([ra, dec, radius_arcsec]).all()
            and 0 <= ra < 360 and -90 <= dec <= 90 and 0 < radius_arcsec <= 180):
        raise ValueError('Need finite RA [0,360), Dec [-90,90], radius (0,180] arcsec.')
    center = SkyCoord(ra*u.deg, dec*u.deg, frame='icrs')
    raw, frames, status = {}, [], []
    for source in ['NED', 'SDSS']:
        try:
            if source == 'NED':
                table = Ned.query_region(center, radius=radius_arcsec*u.arcsec)
                raw[source] = table
                frame = ned_candidates(table, center)
            else:
                table = SDSS.query_region(
                    center, radius=radius_arcsec*u.arcsec, spectro=True,
                    photoobj_fields=['ra', 'dec'],
                    specobj_fields=['specobjid', 'z', 'zErr', 'zWarning', 'class'],
                    data_release=SDSS_RELEASE)
                raw[source] = table
                frame = sdss_candidates(table, center)
            # Recompute the separation, rather than relying on a service's units.
            frame = frame.loc[frame.sep_arcsec <= radius_arcsec + 1e-5].copy()
            frames.append(frame)
            status.append(dict(service=source, status='ok' if len(frame) else 'no_match',
                               n_candidates=len(frame), message=''))
        except Exception as exc:
            message = f'{type(exc).__name__}: {exc}'
            warnings.warn(f'{source} query failed: {message}')
            status.append(dict(service=source, status='error', n_candidates=0, message=message))
    nonempty = [f for f in frames if len(f)]
    candidates = pd.concat(nonempty, ignore_index=True) if nonempty else pd.DataFrame(columns=COLUMNS)
    candidates = candidates.sort_values('sep_arcsec').reset_index(drop=True)
    candidates.index.name = 'candidate_index'
    return candidates, pd.DataFrame(status), raw


In [ ]:
candidates, query_status, raw_tables = query_redshifts(RA, DEC, RADIUS_ARCSEC)
display(query_status)
display(candidates)
# Only candidates with a reported numerical redshift:
display(candidates.loc[pd.to_numeric(candidates.z, errors='coerce').notna()])


## 4. Inspect NED measurements and references (optional)
Choose a NED object from the candidate table by setting `NED_OBJECT_NAME`. This retrieves its redshift measurements, not just its preferred summary redshift. Inspect measurement type, uncertainty, flags and bibliography. NED can contain spectroscopic, photometric or other estimates; do not relabel every NED result as spec-z.

In [ ]:
NED_OBJECT_NAME = "WISEA J095652.35+032811.7"   # e.g. copy the exact NED source_id shown above
if NED_OBJECT_NAME:
    ned_measurements = Ned.get_table(NED_OBJECT_NAME, table='redshifts')
    display(ned_measurements)
    ned_measurements.write(OUT / 'ned_redshift_measurements.ecsv', format='ascii.ecsv', overwrite=True)


## 5. Select a reviewed candidate and save
Nearby stars, background galaxies, knots or AGN may appear in the cone. `zWarning=0` is a useful SDSS quality screen, not proof of a correct positional match. Compare object type, separation and imaging before adoption. NED and SDSS may report the same original measurement.

Leave SELECTED_INDEX=None to save candidates without adopting a redshift. Redshift reference frames and peculiar velocities matter for nearby-galaxy distances; this notebook does not convert heliocentric redshift into a flow-corrected distance.

In [ ]:
SELECTED_INDEX = None       # integer candidate_index after your review
REVIEW_NOTE = ''            # record how identity/type/reference were verified
candidates.to_csv(OUT / 'redshift_candidates.csv', index=True)
query_status.to_csv(OUT / 'query_status.csv', index=False)
for service, table in raw_tables.items():
    if table is not None:
        table.write(OUT / f'{service}_raw.ecsv', format='ascii.ecsv', overwrite=True)
metadata = dict(galaxy_id=SN_ID, ra=RA, dec=DEC, radius_arcsec=RADIUS_ARCSEC,
                sdss_release=SDSS_RELEASE, queried_utc=datetime.now(timezone.utc).isoformat(),
                astroquery_version=version('astroquery'), review_note=REVIEW_NOTE)
(OUT / 'query_metadata.json').write_text(json.dumps(metadata, indent=2))
if SELECTED_INDEX is not None:
    if SELECTED_INDEX not in candidates.index or not REVIEW_NOTE.strip():
        raise ValueError('Select a valid index and write a review note.')
    selected = candidates.loc[[SELECTED_INDEX]].copy()
    if not np.isfinite(float(selected.iloc[0].z)):
        raise ValueError('Selected candidate has no numerical redshift.')
    selected.insert(0, 'SN_ID', SN_ID)
    selected['review_note'] = REVIEW_NOTE
    selected.to_csv(OUT / 'reviewed_redshift.csv', index=False)
    display(selected)
print('Saved to:', OUT.resolve())


## 6. Several galaxies (optional)
Use a DataFrame with `galaxy_id`, `ra`, `dec`. IDs are read as strings to preserve large identifiers. Each input row is retained in the status table even when no candidate exists or coordinates are invalid. The output contains all candidates, not automatically chosen matches. This sequential loop is intended for small samples; for tens of thousands of galaxies use bulk catalog crossmatches.

To adapt an existing table:
```python
galaxies = df.rename(columns={'host_ra':'ra', 'host_dec':'dec', 'diaobjectId':'galaxy_id'})
```
Disable this section for a single target.

In [ ]:
RUN_BATCH = False
INPUT_CSV = Path('galaxies.csv')
if RUN_BATCH:
    galaxies = pd.read_csv(INPUT_CSV, dtype={'SN_id': 'string'})
    assert {'SN_id', 'ra', 'dec'}.issubset(galaxies.columns)
    all_candidates, all_status = [], []
    for input_row, row in galaxies.reset_index(drop=True).iterrows():
        try:
            found, state, _ = query_redshifts(row.ra, row.dec, RADIUS_ARCSEC)
        except (TypeError, ValueError) as exc:
            found = pd.DataFrame(columns=COLUMNS)
            state = pd.DataFrame([dict(service='input', status='invalid_input',
                                       n_candidates=0, message=str(exc))])
        for table in [found, state]:
            table.insert(0, 'galaxy_id', row.galaxy_id)
            table.insert(0, 'input_row', input_row)
            table['input_ra'] = row.ra
            table['input_dec'] = row.dec
        all_candidates.append(found)
        all_status.append(state)
        print(f'{input_row+1}/{len(galaxies)}: {row.galaxy_id}: {len(found)} candidates')
        time.sleep(1)  # modest request rate; no parallel requests
    if all_status:
        batch_status = pd.concat(all_status, ignore_index=True)
        batch_candidates = pd.concat(all_candidates, ignore_index=True)
        batch_status.to_csv(OUT / 'batch_query_status.csv', index=False)
        batch_candidates.to_csv(OUT / 'batch_redshift_candidates.csv', index=False)
        display(batch_candidates)
        display(batch_status.groupby(['service', 'status']).size().rename('count'))


## Troubleshooting and documentation

- `error` in query_status: inspect the message. A timeout or schema failure is not a no-match. Raw tables (if returned) are available in `raw_tables` for debugging.
- `no_match`: check coordinates, footprint and radius. A larger radius increases false matches.
- Missing z: the object exists in the catalog, but no summary redshift is reported.
- SSL certificate failures: repair the Python environment certificates; do not disable verification.
- NED schemas can change across astroquery releases; if a required column is absent the code prints the available columns through its error message.

Sources: [NED astroquery guide](https://astroquery.readthedocs.io/en/latest/ipac/ned/ned.html), [SDSS guide](https://astroquery.readthedocs.io/en/latest/sdss/sdss.html), [SDSS API](https://astroquery.readthedocs.io/en/latest/api/astroquery.sdss.SDSSClass.html).
